# 04b — Refit: does the inflation fix work, and does relative_size earn its place?

Two questions, both testable now that notebook 04 built the features:

1. **Does inflation-adjusting the price fix the Phase 1 baseline's collapse?**
   (train R² 0.452, test R² ≈ 0, test MdAPE 33.6% — traced to pooling 18
   years of rising prices with no time control.)
2. **Does `relative_size` help specifically where plain floor area
   struggles** — uniform-stock streets, per the original buyer's-report
   finding (R² 0.88 on varied stock down to 0.076 on uniform stock)?
   Overall it correlated *more weakly* with price than floor area
   (0.253 vs 0.695), so this is the real test of whether it's worth
   keeping, not just a formality.

Two models, same temporal split as Phase 1 (train up to mid-2024, test
after), so the comparison is apples-to-apples with the original baseline:

- **Model 1:** `log(price_adjusted) ~ log(floor_area)` — same shape as the
  Phase 1 baseline, but on inflation-adjusted prices. Isolates question 1.
- **Model 2:** `log(price_adjusted) ~ log(floor_area) + log(relative_size)`
  — adds the neighbour-comparison feature. Isolates question 2, since any
  difference from Model 1 can only come from that one added feature.


In [1]:
import sys
sys.path.insert(0, "..")
from src.config import DATA_INTERIM
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_parquet(DATA_INTERIM / "ppd_epc_joined_sefton_features.parquet")
df = df.dropna(subset=["total_floor_area", "relative_size", "price_adjusted"]).copy()
df["log_price_adj"] = np.log(df["price_adjusted"])
df["log_area"] = np.log(df["total_floor_area"])
df["log_relsize"] = np.log(df["relative_size"])
print(f"{len(df):,} transactions with everything needed for this fit")


52,928 transactions with everything needed for this fit


In [2]:
split_date = pd.Timestamp("2024-07-01")
train = df[df["date_of_transfer"] < split_date]
test = df[df["date_of_transfer"] >= split_date]
print(f"Train: {len(train):,} | Test: {len(test):,}")

def fit_and_evaluate(feature_cols, label):
    X_train = sm.add_constant(train[feature_cols])
    model = sm.OLS(train["log_price_adj"], X_train).fit()

    X_test = sm.add_constant(test[feature_cols])
    pred_log = model.predict(X_test)
    pred_price = np.exp(pred_log)
    actual = test["price_adjusted"]

    ape = (pred_price - actual).abs() / actual
    mdape = ape.median()
    ppe10 = (ape <= 0.10).mean()
    ss_res = ((test["log_price_adj"] - pred_log) ** 2).sum()
    ss_tot = ((test["log_price_adj"] - test["log_price_adj"].mean()) ** 2).sum()
    test_r2 = 1 - ss_res / ss_tot

    print(f"--- {label} ---")
    print(f"Train R²:  {model.rsquared:.3f}")
    print(f"Test R²:   {test_r2:.3f}")
    print(f"Test MdAPE: {mdape:.1%}")
    print(f"Test PPE10: {ppe10:.1%}")
    print()
    return model, pred_price, {"mdape": mdape, "ppe10": ppe10, "test_r2": test_r2}

model1, pred1, metrics1 = fit_and_evaluate(["log_area"], "Model 1: floor area only (HPI-adjusted)")
model2, pred2, metrics2 = fit_and_evaluate(["log_area", "log_relsize"], "Model 2: floor area + relative_size (HPI-adjusted)")


Train: 46,789 | Test: 6,139
--- Model 1: floor area only (HPI-adjusted) ---
Train R²:  0.482
Test R²:   0.492
Test MdAPE: 20.1%
Test PPE10: 25.5%

--- Model 2: floor area + relative_size (HPI-adjusted) ---
Train R²:  0.542
Test R²:   0.560
Test MdAPE: 18.0%
Test PPE10: 28.4%



## Question 1: did the inflation fix work?

Compare Model 1's test R² against the Phase 1 baseline's **≈ 0**.


In [3]:
print(f"Phase 1 baseline (no HPI adjustment): test R² ≈ 0.00, test MdAPE 33.6%, PPE10 10.0%")
print(f"Model 1 (HPI-adjusted, otherwise identical): test R² = {metrics1['test_r2']:.3f}, "
      f"test MdAPE {metrics1['mdape']:.1%}, PPE10 {metrics1['ppe10']:.1%}")


Phase 1 baseline (no HPI adjustment): test R² ≈ 0.00, test MdAPE 33.6%, PPE10 10.0%
Model 1 (HPI-adjusted, otherwise identical): test R² = 0.492, test MdAPE 20.1%, PPE10 25.5%


## Question 2: does relative_size earn its place?

First the overall comparison (Model 2 vs Model 1), then — the real test —
**stratified by within-area floor-area variance**, the same three tiers
used in notebook 03b. If the hypothesis is right, Model 2 should pull
ahead of Model 1 specifically in the **low-variance** tier, where every
house on the street is a similar size and Model 1 has the least to work
with.


In [4]:
print(f"Model 1 (area only):          test R² = {metrics1['test_r2']:.3f}, MdAPE = {metrics1['mdape']:.1%}")
print(f"Model 2 (area + relative_size): test R² = {metrics2['test_r2']:.3f}, MdAPE = {metrics2['mdape']:.1%}")


Model 1 (area only):          test R² = 0.492, MdAPE = 20.1%
Model 2 (area + relative_size): test R² = 0.560, MdAPE = 18.0%


In [5]:
test_eval = test.copy()
test_eval["postcode_district"] = test_eval["postcode"].str.split(" ").str[0]

# Same tiering approach as notebook 03b: postcode district as the
# neighbourhood unit (Sefton alone doesn't have enough transactions per
# full postcode for a stable variance estimate).
area_variance = df.assign(postcode_district=df["postcode"].str.split(" ").str[0]) \
                   .groupby("postcode_district")["total_floor_area"].std()
test_eval["area_variance_tier"] = pd.qcut(
    test_eval["postcode_district"].map(area_variance), q=3, labels=["low", "medium", "high"]
)

test_eval["pred1"] = pred1.values
test_eval["pred2"] = pred2.values
test_eval["ape1"] = (test_eval["pred1"] - test_eval["price_adjusted"]).abs() / test_eval["price_adjusted"]
test_eval["ape2"] = (test_eval["pred2"] - test_eval["price_adjusted"]).abs() / test_eval["price_adjusted"]

stratified = test_eval.groupby("area_variance_tier", observed=True).agg(
    n=("ape1", "size"),
    model1_mdape=("ape1", "median"),
    model2_mdape=("ape2", "median"),
    model1_ppe10=("ape1", lambda s: (s <= 0.10).mean()),
    model2_ppe10=("ape2", lambda s: (s <= 0.10).mean()),
)
stratified["mdape_improvement"] = stratified["model1_mdape"] - stratified["model2_mdape"]
print(stratified)


                       n  model1_mdape  model2_mdape  model1_ppe10  \
area_variance_tier                                                   
low                 2178      0.206028      0.189699      0.259412   
medium              2009      0.185331      0.167303      0.274764   
high                1952      0.212158      0.183910      0.229508   

                    model2_ppe10  mdape_improvement  
area_variance_tier                                   
low                     0.275941           0.016329  
medium                  0.306620           0.018028  
high                    0.269467           0.028247  


## Read the `mdape_improvement` column

Positive means Model 2 (with `relative_size`) made typical errors
*smaller* in that tier; negative means it made them worse. The hypothesis
predicts the **largest positive number in the "low" row** — if it isn't,
`relative_size` isn't doing the job the plan hoped for and that's worth
knowing now, before it's built into the full Phase 2 feature set.
